# Milestone M1–M2: Exploratory Data Analysis & Problem Framing
## Kelompok 4 — CNN for Text Classification (Indonesian Hate Speech Detection)

**Mata Kuliah:** Workshop Proyek Sistem Cerdas 2026  
**Dosen Pengampu:** Dr. Selvia Ferdiana Kusuma, M.Kom  
**Dataset:** `data/raw/indotoxic2024_annotated_data_v2_final.csv` (28.448 baris) & `data/raw/indotoxic2024_annotator_demographic_data_v2_final.csv` (29 anotator)  

### Agenda & Analisis:
1. **Pemuatan Data Mentah & Demografi Anotator**.
2. **Parsing Label Multi-Annotator** (Majority Voting / Consensus Strategy).
3. **Analisis Imbalance Kelas** (Toxic vs Non-toxic: ~14% vs ~86%).
4. **Analisis Sub-kategori Ujaran Kebencian** (Identity attack, insult, profanity, violence, sexual).
5. **Analisis Panjang Teks & Distribusi Kata** (untuk perancangan `MAX_LEN`).
6. **Inspeksi Bias Demografi Anotator** (Gender, Agama, Etnisitas, Usia).

In [ ]:
import sys
from pathlib import Path
import os
import ast

# Set root path project
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import Config
from src.utils.seed import set_seed

# Set deterministik seed
set_seed(Config.SEED)
print(f"Project Root: {ROOT_DIR}")
print(f"Seed Config : {Config.SEED}")

### 1. Pemuatan Dataset Mentah `indotoxic2024`

In [ ]:
raw_csv_path = ROOT_DIR / Config.RAW_DATA_CSV
demo_csv_path = ROOT_DIR / Config.RAW_DEMOGRAPHIC_CSV

df_raw = pd.read_csv(raw_csv_path)
df_demo = pd.read_csv(demo_csv_path)

print(f"Total baris dataset teks      : {len(df_raw):,} data")
print(f"Total annotator yang terdaftar: {len(df_demo):,} orang")
print("\nKolom dataset mentah:", df_raw.columns.tolist())
df_raw.head(3)

### 2. Ekstraksi dan Agregasi Label Multi-Annotator
Dataset `indotoxic2024` dianotasi oleh multi-anotator per sampel (misal: `['0', '1']`). Kita melakukan agregasi konsensus majority voting (>= 50% persetujuan anotator) untuk menghasilkan label biner `0` (Non-toxic) dan `1` (Toxic).

In [ ]:
def aggregate_annotator_votes(val):
    if isinstance(val, str):
        try:
            arr = ast.literal_eval(val)
            nums = [int(x) for x in arr]
            return 1 if (sum(nums) / len(nums)) >= 0.5 else 0
        except Exception:
            return 0
    return int(val) if not pd.isna(val) else 0

df_raw["label"] = df_raw["toxicity"].apply(aggregate_annotator_votes)

# Ekstraksi sub-kategori
subcategories = [
    "profanity_obscenity",
    "threat_incitement_to_violence",
    "insults",
    "identity_attack",
    "sexually_explicit"
]

for sub in subcategories:
    if sub in df_raw.columns:
        df_raw[f"sub_{sub}"] = df_raw[sub].apply(aggregate_annotator_votes)

print("Sampel hasil parsing label:")
df_raw[["text_id", "text", "toxicity", "label"] + [f"sub_{s}" for s in subcategories]].head(5)

### 3. Distribusi Target Label Utama (Class Imbalance)

In [ ]:
label_counts = df_raw["label"].value_counts()
label_percentages = df_raw["label"].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({
    "Jumlah Sampel": label_counts,
    "Persentase (%)": label_percentages.round(2)
})
dist_df.index = ["Non-toxic (0)", "Toxic (1)"]
print(dist_df)

plt.figure(figsize=(7, 4))
ax = sns.barplot(x=dist_df.index, y=dist_df["Jumlah Sampel"], palette=["#27AE60", "#E74C3C"])
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,} ({p.get_height()/len(df_raw)*100:.1f}%)',
                (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.title("Distribusi Kelas Biner (indotoxic2024 Raw)")
plt.ylabel("Frekuensi")
plt.tight_layout()
plt.show()

### 4. Distribusi Sub-Kategori Ujaran Kebencian

In [ ]:
sub_sums = {s: df_raw[f"sub_{s}"].sum() for s in subcategories}
sub_df = pd.DataFrame(list(sub_sums.items()), columns=["Sub-Kategori", "Total Terdeteksi"])
sub_df["Persentase dari Total Data (%)"] = (sub_df["Total Terdeteksi"] / len(df_raw) * 100).round(2)
sub_df = sub_df.sort_values(by="Total Terdeteksi", ascending=False)

print(sub_df)

plt.figure(figsize=(9, 4))
sns.barplot(data=sub_df, y="Sub-Kategori", x="Total Terdeteksi", palette="Reds_r")
plt.title("Frekuensi Sub-Kategori Toxic pada indotoxic2024")
plt.xlabel("Jumlah Kasus")
plt.tight_layout()
plt.show()

### 5. Analisis Karakteristik Panjang Teks & Penentuan `MAX_LEN`

In [ ]:
df_raw["char_count"] = df_raw["text"].astype(str).apply(len)
df_raw["word_count"] = df_raw["text"].astype(str).apply(lambda x: len(x.split()))

print("Statistik Deskriptif Jumlah Kata per Dokumen:")
percentiles = [0.50, 0.75, 0.90, 0.95, 0.99]
print(df_raw["word_count"].describe(percentiles=percentiles))

coverage_128 = (df_raw["word_count"] <= Config.MAX_LEN).mean() * 100
print(f"\nCakupan sekuens dengan MAX_LEN = {Config.MAX_LEN}: {coverage_128:.2f}% dari seluruh dataset")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df_raw["word_count"], bins=40, kde=True, ax=axes[0], color="#2980B9")
axes[0].axvline(Config.MAX_LEN, color="red", linestyle="--", label=f"MAX_LEN ({Config.MAX_LEN})")
axes[0].set_title("Distribusi Jumlah Kata")
axes[0].legend()

sns.boxplot(x=df_raw["label"], y=df_raw["word_count"], ax=axes[1], palette=["#27AE60", "#E74C3C"])
axes[1].set_title("Jumlah Kata: Non-toxic vs Toxic")
axes[1].set_xticklabels(["Non-toxic (0)", "Toxic (1)"])
axes[1].set_ylim(0, 150)
plt.tight_layout()
plt.show()

### 6. Analisis Demografi Anotator (Etika & Bias Evaluasi)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.countplot(data=df_demo, y="gender", ax=axes[0, 0], palette="Blues_d")
axes[0, 0].set_title("Distribusi Gender Anotator")

sns.countplot(data=df_demo, y="religion", ax=axes[0, 1], palette="Greens_d")
axes[0, 1].set_title("Distribusi Agama Anotator")

sns.countplot(data=df_demo, y="age", ax=axes[1, 0], palette="Purples_d")
axes[1, 0].set_title("Distribusi Usia Anotator")

sns.countplot(data=df_demo, y="ethnicity", ax=axes[1, 1], palette="Oranges_d")
axes[1, 1].set_title("Distribusi Etnis Anotator")

plt.tight_layout()
plt.show()

### 7. Kesimpulan & Temuan Kunci M1–M2
1. **Ukuran Korpus Nyata:** 28.448 baris tweet/post media sosial Indonesia dengan 29 anotator beraneka latar belakang demografi.
2. **Severe Class Imbalance:** Kelas Toxic (14,04%) vs Non-toxic (85,96%). Rasio ketidakseimbangan mencapai ~1 : 6. Hal ini menuntut penggunaan `class_weight`, `focal_loss`, atau augmentasi di tahap M6–M7.
3. **Kategori Toksik Dominan:** Kategori ujaran kebencian tertinggi adalah *Insults* (6,84%) dan *Identity Attack* (6,17%).
4. **Parameter Sequencing:** `MAX_LEN = 128` terbukti mencakup >98% dari panjang dokumen aktual tanpa informasi yang terpangkas secara signifikan.